# Lab 08.2 — CloudWatch Alarms for Governance

## Overview

8 alarmes para detectar comportamentos anômalos:
- DENY rate alto (possível user error ou ataque)
- Runtime errors (falha de agente)
- Gateway high latency (incident)
- Guardrail blocks (comportamento abusivo)

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
from shared.utils.config import load_config, save_config, get_region
from utils import create_governance_alarms

cfg = load_config()
region = get_region()

## Step 1: Criar alarmes (sem SNS por enquanto)

São **8 alarmes** mapeados aos planos de controle do workshop:

| Alarme | Namespace / Mistrica | Dispara when |
|---|---|---|
| `workshop-CedarDenySpike` | AWS/Bedrock-AgentCore · DenyDecisions | ≥ 10 DENYs do Cedar em 5 min |
| `workshop-LambdaErrors-*` (5×) | AWS/Lambda · Errors | ≥ 5 erros em 5 min em cada tool Lambda |
| `workshop-GuardrailIntervened` | AWS/Bedrock/Guardrails · InvocationsIntervened | ≥ 5 intervenções do guardrail em 5 min |
| `workshop-RuntimeSyshasErrors` | AWS/Bedrock-AgentCore · SyshasErrors | ≥ 3 erros 5xx do Runtime em 5 min |

> 💡 As mistricas só aparecem after tráfego real; atis lá os alarmes ficam
> em `INSUFFICIENT_DATA` (normal).

In [ ]:
import boto3
account_id = boto3.client("sts").get_caller_identity()["Account"]
lambda_names = [cfg[k].split(":")[-1] for k in (
    "LAMBDA_GRID_ARN", "LAMBDA_MAINTENANCE_ARN", "LAMBDA_CONTRACT_ARN",
    "LAMBDA_BILLING_ARN", "LAMBDA_REGULATORY_ARN",
) if cfg.get(k)]

alarms = create_governance_alarms(
    policy_engine_id=cfg.get("POLICY_STORE_ID"),
    guardrail_id=cfg.get("BEDROCK_GUARDRAIL_ID"),
    guardrail_version=cfg.get("BEDROCK_GUARDRAIL_VERSION", "DRAFT"),
    lambda_function_names=lambda_names,
    account_id=account_id,
    region=region,
)
print(f"\n{len(alarms)} alarmes criados")

## Step 2 (optional): Create SNS topic and re-create alarms with notification

In [ ]:
# import boto3
# sns = boto3.client("sns", region_name=region)
# topic = sns.create_topic(Name="workshop-governance-alarms")
# sns.subscribe(TopicArn=topic["TopicArn"], Protocol="email", Endpoint="seu@email.com")
# save_config({"SNS_ALARM_TOPIC_ARN": topic["TopicArn"]})
# alarms = create_governance_alarms(sns_topic_arn=topic["TopicArn"], region=region)
print("To receive email notification, uncomment the lines above")

## 🎓 What you learned

- 8 alarms cover the main risks: authorization, latency, errors, abuse
- SNS provides proactive notification
- In production: integrar com PagerDuty, Slack, etc.

## Cleanup

```python
from utils import cleanup_alarms, cleanup_cloudtrail
cleanup_alarms(region=region)
cleanup_cloudtrail("workshop-trail", region=region)
```

## Next

➡️ [Lab 09 — End-to-End com UI](../09-End-to-End-with-UI/)